<a href="https://colab.research.google.com/github/zeroKool1ne/ironhack_langchain/blob/main/3.2_create_llm_agent_with_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create an LLM Agent with LangChain

We are going to create an Agent With LangChain that using the OpenAI API, will be able to analyze the data contained in an Excel file.

It will be able to find relationships between variables, clean the data, search for a model, and execute it to make future predictions.

In summary, it will act as a Data Scientist Assistant, helping us in our day-to-day tasks.

## Intalling and importing libraries

In [2]:
!pip install langchain langchain_experimental langchain-classic langchain-anthropic anthropic langchain-community
# !pip install --upgrade pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


We use the **os** library to store Environ variables. Like OPENAI_API_KEY.

Get you OpenAI API  Key: https://platform.openai.com/

In [14]:
from google.colab import userdata
import os

os.environ["CLAUDE_API_KEY"] = userdata.get("claude_ironhack")

## Loading the Data

In [6]:
# from google.colab import drive
# drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
#!pip install kaggle

In [ ]:
# import os
# #This directory should contain you kaggle.json file with you key
# os.environ['KAGGLE_CONFIG_DIR'] = '/content/drive/MyDrive/kaggle'

###  Upload `kaggle.json` to Colab

First, create a [Kaggle](https://www.kaggle.com/) account if you don't have one.

Then, download `kaggle.json` from [Kaggle](https://www.kaggle.com/) → *Account* → *Create New API Token*.

In Colab:
- Click on the **Files** icon (left sidebar).
- Click the **Upload** button.
- Upload your `kaggle.json` file.

In [ ]:
# import shutil
# shutil.move("/content/kaggle.json", "/content/drive/MyDrive/kaggle")


In [ ]:
# !kaggle datasets download -d goyaladi/climate-insights-dataset

In [7]:

# import zipfile

# # Define the path to your zip file
# file_path = '/content/climate-insights-dataset.zip'
# with zipfile.ZipFile(file_path, 'r') as zip_ref:
#     zip_ref.extractall('/content/drive/MyDrive/kaggle')

import kagglehub

# Download latest version
path = kagglehub.dataset_download("goyaladi/climate-insights-dataset")

print("Path to dataset files:", path)


100%|██████████| 791k/791k [00:00<00:00, 87.2MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/goyaladi/climate-insights-dataset/versions/4


We are using a kaggle Dataset:
https://www.kaggle.com/datasets/goyaladi/climate-insights-dataset

You can download the CSV. Feel free to use the Dataset you are more interested In, or your own Data.




In [8]:
import pandas as pd
csv_file = path + '/climate_change_data.csv'
#creating the document with Pandas.
document = pd.read_csv(csv_file)

In [9]:
document.head(5)

,Date,Location,Country,Temperature,CO2 Emissions,Sea Level Rise,Precipitation,Humidity,Wind Speed
0,2000-01-01 00:00:00.000000000,New Williamtown,Latvia,10.688986,403.118903,0.717506,13.835237,23.631256,18.492026
1,2000-01-01 20:09:43.258325832,North Rachel,South Africa,13.814430,396.663499,1.205715,40.974084,43.982946,34.249300
2,2000-01-02 16:19:26.516651665,West Williamland,French Guiana,27.323718,451.553155,-0.160783,42.697931,96.652600,34.124261
3,2000-01-03 12:29:09.774977497,South David,Vietnam,12.309581,422.404983,-0.475931,5.193341,47.467938,8.554563
4,2000-01-04 08:38:53.033303330,New Scottburgh,Moldova,13.210885,410.472999,1.135757,78.695280,61.789672,8.001164


# Creating the Agent
This is the easiest Agent we can create with LangChain, we only need to import the **create_pandas_dataframe_agent**.

Time to create our little assistant, and we need only a call.

We let **OpenAI** decide which model to use. However, we specify a **temperature** value of 0 to its parameter, so that it is not imaginative. This is much better when we want the model to be able to give commands to the different libraries it can use.



In [10]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_anthropic import ChatAnthropic

/tmp/ipykernel_410/2809198245.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent


In [19]:
sm_ds_OAI = create_pandas_dataframe_agent(
    ChatAnthropic(
        model="claude-haiku-4-5",
        api_key=CLAUDE_API_KEY,
        temperature=0
    ),
    document,
    verbose=True,
    allow_dangerous_code=True,
    handle_parsing_errors=True,
)

/usr/local/lib/python3.12/dist-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


## First Question.

In [20]:
sm_ds_OAI.invoke("Please conduct a basic EDA and tell me the following metrics in tabular form: shape, missing values, basic statistics, data types, column names and also what the dataset is about and if there are any outliers or anomalies.")



> Entering new AgentExecutor chain...


ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `I'll conduct a comprehensive EDA on the dataframe. Let me start by exploring its structure and contents.
<function_calls>
<invoke name="python_repl_ast">
<parameter name="code">
import pandas as pd
import numpy as np

# First, let's get the shape
print("=" * 80)
print("DATASET SHAPE")
print("=" * 80)
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
</parameter>
</invoke>
</function_calls>
<function_calls>
<invoke name="python_repl_ast">
<parameter name="code">
# Column names and data types
print("\n" + "=" * 80)
print("COLUMN NAMES AND DATA TYPES")
print("=" * 80)
print(df.dtypes)
</parameter>
</invoke>
</function_calls>
<function_calls>
<invoke name="python_repl_ast">
<parameter name="code">
# Missing values
print("\n" + "=" * 80)
print(

In [ ]:
document.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            10000 non-null  str    
 1   Location        10000 non-null  str    
 2   Country         10000 non-null  str    
 3   Temperature     10000 non-null  float64
 4   CO2 Emissions   10000 non-null  float64
 5   Sea Level Rise  10000 non-null  float64
 6   Precipitation   10000 non-null  float64
 7   Humidity        10000 non-null  float64
 8   Wind Speed      10000 non-null  float64
dtypes: float64(6), str(3)
memory usage: 1.2 MB


The description of the data made by the Agent is acurated.

## Second Question.

In [ ]:
# sm_ds_OAI.run("Do you think is possible to forecast the temperature?")


The model thinks that is possible to forecast the temperature, but difficult because the weak correlation between variables.

I don't now why the model decided to create a graphic bar.

## Third question

In [ ]:
# sm_ds_OAI.run("""
# Can you create a line graph containing the anual average co2 emissions over the years?
# """)


## Fourth question

In [ ]:
# sm_ds_OAI.run("""
# Create a line graph with seaborn containing the anual average co2 emissions in Portugal over the years.
# """)

## Last Question.

In [ ]:
# sm_ds_OAI.invoke("""Select a forecasting model to forecast the temperature.
# Use this kind of model to forecast the average temperature
# for year in Port Maryberg in Malta for the next 5 years.
# Write the temperatures forecasted in a table.""")

Note how the agent is capable of installing the necessary software for the actions it needs to perform.

# Conclusions

This is one of the most powerful and, at the same time, easiest to use agents. We have seen how with just a few lines of code, we had an agent capable of following our instructions to analyze, clean, and generate charts from our data. Not only that, but it has also been able to draw conclusions and even decide which algorithm was best for forecasting the data.

The world of agents is just beginning, and many players are entering the field, such as Hugging Face, Microsoft, or Google. Their capabilities are not only growing with new tools but also with new language models.

**It's a revolution that we cannot afford to miss and will change many things.**